# 🔐 第一课 · 密码学地基

## 本课目标

学完这一课你会理解：
- 比特币用的哈希函数 SHA-256 是什么、为什么安全
- "挖矿找前导零"到底在找什么
- 比特币地址是怎么从公钥一步步算出来的
- 椭圆曲线签名为什么能证明"这笔钱是我的"

> 不需要任何密码学背景。我们用 Python 代码来感受每一个概念。

## 1. 什么是哈希函数？

想象你有一台**绞肉机**：

- 放进去一整头牛 → 出来一碗肉馅
- 放进去一只鸡 → 出来一碗肉馅
- 放进去一根胡萝卜 → 也出来一碗肉馅

绞肉机有四个特性：
1. **输出大小固定**——不管输入多大，输出都是一碗
2. **同输入同输出**——同一头牛绞两次，出来的肉馅一模一样
3. **绞了就回不来**——给你一碗肉馅，你能反推出原来是什么动物吗？不能
4. **没有两头动物绞出来一样**——找不到两头不同的动物绞出完全一样的肉馅

SHA-256 就是这样一个"数字绞肉机"：
- 输入可以是 1 字节，也可以是整部电影
- 输出永远是 **256 位**（32 字节，64 个十六进制字符）

In [ ]:
import hashlib

# SHA-256：输入什么都可以，输出固定 32 字节
def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

# 演示：三个不同大小的输入，输出都是 32 字节
for label, data in [
    ("1 个字符", b"a"),
    ("一句话", b"Hello, Bitcoin!"),
    ("一段文字", b"The quick brown fox jumps over the lazy dog" * 10),
]:
    h = sha256(data)
    print(f"{label:8s} → {len(data):4d} 字节输入 → {len(h)} 字节输出")
    print(f"           哈希: {h.hex()[:40]}...")
    print()

## 2. 雪崩效应 — 差一点，全不同

SHA-256 最神奇的特性：**改输入的一个比特，输出的一半比特都会翻转**。

这一点对挖矿至关重要——意味着你无法"猜"出一个 nonce 来靠近目标。每次改 nonce，哈希结果都完全不同，和之前的结果毫无关系。**只能暴力试。**

下面我们来亲眼看看这个效应：

In [ ]:
def avalanche_demo():
    # 两条消息，只差最后一个字母的大小写
    a = b"hello world"
    b = b"hello worlD"

    h1 = sha256(a)
    h2 = sha256(b)

    print(f'  "hello world" → {h1.hex()}')
    print(f'  "hello worlD" → {h2.hex()}')
    print()

    # 逐字节比较：有多少字节不同？
    same_bytes = sum(1 for x, y in zip(h1, h2) if x == y)
    diff_bytes = len(h1) - same_bytes
    print(f"  32 字节中 {same_bytes} 相同, {diff_bytes} 不同")
    print(f"  约 {diff_bytes/32*100:.0f}% 的字节发生了变化")
    print()
    print("  💡 只改了一个字母的大小写，结果面目全非。")
    print("  💡 这就是为什么挖矿只能暴力试——无法预测")

avalanche_demo()

## 3. 双重 SHA-256

比特币实际上不使用单次 SHA-256，而是做两次：

```
block_hash = SHA-256( SHA-256( block_header ) )
```

叫 **double SHA-256**。中本聪这么做的原因有几种说法（防御长度扩展攻击、增加安全性），但最实际的效果是：让比特币和其他用单 SHA-256 的系统区分开来。

In [ ]:
def double_sha256(data: bytes) -> bytes:
    # 比特币实际使用的双 SHA-256
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

# 比较单次和双次
data = b"Bitcoin block header data"
print(f"单次: {sha256(data).hex()}")
print(f"双次: {double_sha256(data).hex()}")

## 4. 前导零 — "挖矿"就是找这个

比特币挖矿的核心要求：区块头的 double SHA-256 结果必须**小于某个目标值**。

因为哈希值是一个 256 位的大整数，**小于某个目标** ≈ **结果前面有很多个 0**。

下面我们亲手体验：给定数据，不断加 nonce（一个计数器），直到哈希以指定数量的零开头。

In [ ]:
import time

def mine_zeros(target_zeros: int):
    # 找 nonce 使得哈希以 target_zeros 个零字节开头
    nonce = 0
    target_prefix = b"\x00" * target_zeros  # 目标：前 N 个字节都是 0
    start = time.time()

    while True:
        # 把数据和 nonce 拼起来做哈希
        guess = sha256(f"block:{nonce}".encode())
        if guess.startswith(target_prefix):
            break
        nonce += 1

    elapsed = time.time() - start
    print(f"  要求前 {target_zeros} 个零字节 ({target_zeros*8} 个零比特)")
    print(f"  试了 {nonce:,} 次")
    print(f"  耗时 {elapsed:.3f} 秒")
    print(f"  算力 ≈ {nonce/elapsed:,.0f} 哈希/秒")
    print(f"  找到的哈希: {guess.hex()}")
    print()

# 试试不同难度
for z in [1, 2, 3]:
    mine_zeros(z)

print("💡 每多一个零字节，难度 × 256")
print("💡 比特币当前难度要求约 17 个前导零字节！")

## 5. 从公钥到比特币地址

比特币地址不是你的公钥本身，而是经过多层哈希和编码的结果。流程如下：

```
公钥 → SHA-256 → RIPEMD-160 → 加版本字节 → 双SHA256取校验 → Base58编码
```

每一步的目的：
- **RIPEMD-160**：把 256 位哈希压缩到 160 位，地址更短
- **版本字节**：区分主网(0x00 → 地址以1开头)和测试网
- **校验和**：防止抄错地址（类似身份证最后一位校验码）
- **Base58**：去掉容易混淆的字符（0/O/I/l），地址美观易读

In [ ]:
def hash160(data: bytes) -> bytes:
    # Bitcoin 的 HASH160 = RIPEMD-160(SHA-256(data))
    sha = hashlib.sha256(data).digest()
    rip = hashlib.new("ripemd160")
    rip.update(sha)
    return rip.digest()

def base58_encode(data: bytes) -> str:
    # Base58 编码（不含 0OIl 等易混淆字符）
    alphabet = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
    n = int.from_bytes(data, "big")
    result = []
    while n > 0:
        n, rem = divmod(n, 58)
        result.append(alphabet[rem])
    # 前导零字节 → 前导 '1'
    for b in data:
        if b == 0:
            result.append("1")
        else:
            break
    return "".join(reversed(result))

def pubkey_to_address(pubkey: bytes) -> str:
    # 从公钥生成 P2PKH 比特币地址（以 '1' 开头）
    h160 = hash160(pubkey)
    payload = b"\x00" + h160  # 主网版本字节 0x00
    checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
    return base58_encode(payload + checksum)

# 用一个随机"公钥"演示
import os
demo_pk = os.urandom(33)
addr = pubkey_to_address(demo_pk)
print(f"公钥 (hex): {demo_pk.hex()[:40]}...")
print(f"比特币地址:  {addr}")
print()
print("💡 这就是你给别人转币时填的地址")
print("💡 地址以 '1' 开头 = 传统 P2PKH 地址")

## 6. 椭圆曲线数字签名 (ECDSA)

比特币用 **secp256k1** 椭圆曲线来签名交易。核心逻辑：

- **私钥**：一个随机的大整数（256 位），你保密
- **公钥**：私钥 × 曲线上的生成点 G，可以公开
- **签名**：用私钥对交易数据签名
- **验证**：用公钥验证签名是否合法

**安全性**建立在"椭圆曲线离散对数问题"上：知道 G 和 k·G（公钥），算不出 k（私钥）。

下面用一个极小的椭圆曲线来演示原理（真正的 secp256k1 用的是 256 位的超大参数）：

In [ ]:
# 一个极小的椭圆曲线：y^2 = x^3 + 2x + 3 (mod 97)
# 仅用于演示原理，不是安全的曲线！

class TinyCurve:
    def __init__(self, a, b, p, Gx, Gy, n):
        self.a, self.b, self.p = a, b, p
        self.G = (Gx, Gy)    # 生成点
        self.n = n            # 曲线的"阶"

    def modinv(self, k):
        # 模逆元：k * k^(-1) ≡ 1 (mod p)
        return pow(k, -1, self.p)

    def point_add(self, P, Q):
        # 椭圆曲线上的点加法
        if P is None: return Q
        if Q is None: return P
        x1, y1 = P
        x2, y2 = Q
        # 如果两点对称于 x 轴，和为无穷远点
        if x1 == x2 and (y1 + y2) % self.p == 0:
            return None
        if P == Q:
            # P+P = 2P：用切线斜率
            lam = (3 * x1 * x1 + self.a) * self.modinv(2 * y1) % self.p
        else:
            # P+Q：用两点连线斜率
            lam = (y2 - y1) * self.modinv(x2 - x1) % self.p
        x3 = (lam * lam - x1 - x2) % self.p
        y3 = (lam * (x1 - x3) - y1) % self.p
        return (x3, y3)

    def scalar_mult(self, k, P=None):
        # 标量乘法: k × P（用快速幂算法，O(log k)）
        if P is None:
            P = self.G
        result = None
        addend = P
        while k:
            if k & 1:  # 当前比特是 1
                result = self.point_add(result, addend)
            addend = self.point_add(addend, addend)  # 翻倍
            k >>= 1
        return result

# 创建一条极小曲线
curve = TinyCurve(a=2, b=3, p=97, Gx=3, Gy=6, n=100)

# 随机生成私钥，算出公钥
import random
private_key = random.randint(1, curve.n - 1)
public_key = curve.scalar_mult(private_key)

print(f"曲线: y^2 = x^3 + {curve.a}x + {curve.b} (mod {curve.p})")
print(f"生成点 G = {curve.G}")
print()
print(f"私钥: {private_key}  ← 只有你知道")
print(f"公钥: {public_key}  ← 可以公开")
print(f"验证: 私钥×G = {curve.scalar_mult(private_key)} == 公钥? {curve.scalar_mult(private_key) == public_key}")
print()
print("💡 真正的 secp256k1 参数是 256 位的超大质数")
print("💡 如果你能反推出私钥，你就破解了比特币")

## 本课小结

| 概念 | 记住它 |
|------|--------|
| SHA-256 | 输入任意 → 输出固定 256 位，"数字指纹" |
| 雪崩效应 | 改一点，全盘变。挖矿只能暴力试，不能猜 |
| 双 SHA-256 | 比特币特有：sha256(sha256(data)) |
| 前导零 | 挖矿的目标 = 让哈希前面有很多个 0 |
| HASH160 | 比特币地址生成的核心步骤 |
| ECDSA | 私钥→公钥不可逆，是比特币安全的数学基础 |

**下一课**：把这些拼成"工作量证明"，理解挖矿的完整数学逻辑。